> 🚨 **[Warning] 무단 도용, 복제 및 배포 금지 안내**
>
> 저작권법에 따라 강의에 사용된 모든 저작물 (코드, 프롬프트, PDF, 실습자료 등)을  
> 무단 복제하거나 외부에 유출할 경우 **_법적 문제가 발생할 수 있습니다._**


# 📂 <font color='#1A4BC0'><b>Part 04. 프롬프트 제작 & 고급 기법</b></font>

## <font color='Darkorange'><b>[ Chapter 01 ]</b></font> 프롬프트 제작 기초
해당 챕터는 **주피터 노트북 실습 기반**으로 진행됩니다.  
실습 시작 전 아래 설정을 반드시 실행해주세요.



```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.

다시 실행해주세요.
```

### ⚙️ <font color='#007A45'><b>[ 실습 전 ]</b></font> Part4 Chapter 01 실습 전 프로젝트 셋업
>  ✅ 아래 **실습 전 가상환경을 활성화하고, 프로젝트 셋업**을 완료한 후 본 실습을 진행해주세요.

> ⚠️ 실습 진행 중 에러가 발생하거나, 세션이 종료되어 런타임이 재시작된 경우, 이 블럭을 항상 다시 실행해주세요.


```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.
```

#### 실습 진행을 위한 라이브러리 다운로드

실습을 진행하기 위해서는 각 AI서비스들의 라이브러리들을 설치해야합니다.
아래 코드 블럭을 실행해서 라이브러리를 설치해봅시다!

```
💡 앞으로 아래 블럭과 같은 코드 블럭은 해당 블럭을 클릭하신 다음 왼쪽의 실행버튼(▶️)을 클릭하거나, `shift + Enter` 단축키를 통해 실행합니다.
```

In [ ]:
# 필요한 패키지 설치 (최초 1회만)
%pip install -r ../requirements.txt

#### 실습 진행을 위한 API KEY 세팅

실습을 진행하기 위해서는 각 AI서비스들의 API Key를 발급 및 세팅 해야합니다.

LangSmith, Gemini, Claude, Chat GPT API Key를 모두 발급하셨다면, 아래 코드 블럭을 실행하여 API Key를 세팅해봅시다.


In [ ]:
# LangSmith & OpenAI Key 설정
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# API KEY 정보 로드
load_dotenv()

# ChatOpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

print("✅ 환경 설정 완료!")
print(f"사용 모델: gpt-4o-mini")
response = model.invoke("안녕하세요?")
print(response.content)

#### 실습 진행을 위해 모델 호출 함수 정의 세팅

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 기본 모델 생성
models = {
    "openai": ChatOpenAI(model="gpt-4o-mini"),
    "claude": ChatAnthropic(model="claude-3-5-haiku-20241022"),
    "gemini": ChatGoogleGenerativeAI(model="gemini-2.0-flash"),
}

parser = StrOutputParser()


# 베이스 체인
def _run_base_chain(model_obj, system_prompt=None, user_input=None, **overrides):
    if overrides:
        model_obj = model_obj.with_config(**overrides)

    model_name = str(type(model_obj)).lower()

    if "claude" in model_name:
        if not user_input and system_prompt:
            user_input = system_prompt
            system_prompt = None

    messages = []
    if system_prompt:
        messages.append(("system", system_prompt))
    if user_input:
        messages.append(("user", user_input))

    if not messages:
        raise ValueError("Claude requires at least one user or system message.")

    prompt = ChatPromptTemplate.from_messages(messages)
    chain = prompt | model_obj | parser
    return chain.invoke({})


# 모델별 메인 실행 체인
def run_openai_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["openai"], system_prompt, user_input, **kwargs)


def run_claude_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["claude"], system_prompt, user_input, **kwargs)


def run_gemini_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["gemini"], system_prompt, user_input, **kwargs)

In [ ]:
from langchain.schema import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver


def create_chat_graph(provider="openai", system_prompt=None):
    model = models[provider]

    builder = StateGraph(MessagesState)

    def call_model(state):
        """state에서 messages를 추출해 모델을 호출하고, 응답 메시지를 반환"""
        messages = state["messages"]

        if system_prompt and not any(isinstance(m, SystemMessage) for m in messages):
            messages = [SystemMessage(content=system_prompt)] + messages

        response = model.invoke(messages)
        return {"messages": messages + [response]}

    builder.add_node("chatbot", call_model)
    builder.set_entry_point("chatbot")

    checkpointer = MemorySaver()
    return builder.compile(checkpointer=checkpointer)

실행 예시를 알아봅시다.

In [ ]:
# 실행 예시
system_prompt = "You are a concise and helpful AI assistant."
user_input = "너에 대해 소개해줘!"

# OpenAI 모델 실행
print("# OpenAI Result:")
print(run_openai_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Claude 모델 실행
print("# Claude Result:")
print(run_claude_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Gemini 모델 실행
print("# Gemini Result:")
print(run_gemini_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

파라미터 수정 예시를 알아봅시다.

In [ ]:
# 1. Temperature (창의성 조절)
print("# OpenAI (temperature=0.8)")
print(
    run_openai_chain(
        system_prompt=system_prompt, user_input=user_input, temperature=0.8
    )
)
print("-" * 40)


# 2. Top-p
print("# Claude (top_p=0.7)")
print(run_claude_chain(system_prompt=system_prompt, user_input=user_input, top_p=0.7))
print("-" * 40)


# 3. Max tokens (출력 길이 제한)
print("# Gemini (max_output_tokens=100)")
print(
    run_gemini_chain(
        system_prompt=system_prompt, user_input=user_input, max_output_tokens=100
    )
)
print("-" * 40)


# 4. 모델 이름 교체
print("# OpenAI (model='gpt-3.5-turbo')")
print(
    run_openai_chain(
        system_prompt=system_prompt, user_input=user_input, model="gpt-3.5-turbo"
    )
)
print("-" * 40)


# 5. 복수 파라미터 동시 변경
print("# Claude (temperature=0.9, top_p=0.95)")
print(
    run_claude_chain(
        system_prompt=system_prompt, user_input=user_input, temperature=0.9, top_p=0.95
    )
)
print("-" * 40)

#### 실습 확인을 위한 LangSmith 추적 세팅 함수

사용자가 실습 기록을 구분하기 위해 LangSmith 프로젝트명을 입력하면 되는 함수입니다.

입력한 이름으로 LangSmith 대시보드에 실행 내역이 저장됩니다.
(예: prompt-course, rag-lab1, myproject-001 등)


```python
# 프로젝트명을 변수로 바로 지정
LANGSMITH_PROJECT = "prompt-course"

# 함수 호출로 환경변수 등록
setup_langsmith(LANGSMITH_PROJECT)
```



In [ ]:
# LangSmith 설정 함수 (프로젝트명만 입력받아 환경변수 등록)


def setup_langsmith(project_name: str):
    """
    LangSmith 관련 환경변수를 등록하는 함수입니다.
    이미 등록된 LANGSMITH_API_KEY를 사용하며,
    project_name 변수로 LangSmith 프로젝트명을 지정할 수 있습니다.
    """
    LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
    LANGSMITH_TRACING = "true"

    os.environ.update(
        {
            "LANGSMITH_PROJECT": project_name,
            "LANGSMITH_ENDPOINT": LANGSMITH_ENDPOINT,
            "LANGSMITH_TRACING": LANGSMITH_TRACING,
        }
    )

    print("✅ LangSmith 설정 완료")
    print(f"- PROJECT : {project_name}")
    print(f"- ENDPOINT: {LANGSMITH_ENDPOINT}")
    print(f"- TRACING : {LANGSMITH_TRACING}")

#### 최종 실습 준비

In [ ]:
setup_langsmith("prompt-course")

### <font color='Saddlebrown'><b>[ 이론 ]</b></font> 프롬프트 기본요소 이해하기
- Instruction, Context, Input, Output

> 해당 파트는 **슬라이드 기반**으로 설명합니다.

> 패스트캠퍼스 온라인 수강 환경에서 슬라이드를 다운로드 받아주세요.

### <font color='green'><b>[ 실습 ] </b></font> 프롬프트 Structure 이해하기
> system, asisstant, user, developer / template-driven prompting


▶︎ **실습문제**: **다음 프롬프트를 기호를 사용하여 구분해보세요.**

<hr>


**실습할 프롬프트 예시**:
```
User:로 시작하여 프롬프트를 작성하세요.
Claude에게 어떤 역할을 맡아야 하는지, 달성해야 할 목표나 주요 작업이 무엇인지에 대한 맥락(context) 을 제공하세요.
상호작용에서 중요하다면, Claude가 사용할 어조(tone) 도 지정할 수 있습니다.
Claude에게 수행해야 할 구체적인 작업 내용을 확장하여 설명하고,Claude가 반드시 따라야 할 규칙(rules) 을 명시하세요.
또한 Claude가 답을 모를 경우 어떻게 행동해야 하는지(예: “모를 경우 이렇게 말해라”) 에 대한 지침도 이 부분에 포함할 수 있습니다.
Claude에게 따라할 수 있는 이상적인 응답(example response) 을 하나 이상 제공하세요.이 예시는 <example></example> XML 태그로 감싸야 합니다.
여러 개의 예시를 제공해도 괜찮습니다.
만약 여러 예시를 제공한다면, 각 예시가 어떤 상황 또는 목적의 예시인지에 대한 맥락(context) 을 Claude에게 설명하고, 각 예시를 자신만의 <example></example> 태그 쌍으로 구분하여 작성하세요.
```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=system_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=system_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=system_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='Saddlebrown'><b>[ 이론 ]</b></font> Delimiters 사용팁 (XML, Markdown, JSON 등)

> 해당 파트는 **슬라이드 기반**으로 설명합니다.

> 패스트캠퍼스 온라인 수강 환경에서 슬라이드를 다운로드 받아주세요.



### <font color='green'><b>[ 실습 ] </b></font> 기초 기법: Shot prompting (0/1/few-shot), Chain-of-Thought






▶︎ **실습문제**: **Translation_a**

<hr>

🎯 **제작 조건:**
  1. 텍스트에서 기술 용어는 원문의 영어를 살려, 영어(한국어 뜻) 이 나오게 하기
  2. 기술 용어를 제외한 나머지 텍스트는 한국어로 번역하기


<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

In [ ]:
# 데이터셋 정의 -- 교재 실습자는 이 셀을 편집하지 마십시오.
data = """
The o1 model series is trained with large-scale reinforcement learning to reason using chain of thought.
These advanced reasoning capabilities provide new avenues for improving the safety and robustness of our models.
In particular, our models can reason about our safety policies in context when responding to potentially unsafe prompts.
This leads to state-of-the-art performance on certain benchmarks for risks such as generating illicit advice, choosing stereotyped responses, and succumbing to known jailbreaks.
Training models to incorporate a chain of thought before answering has the potential to unlock substantial benefits, while also increasing potential risks that stem from heightened intelligence.
Our results underscore the need for building robust alignment methods, extensively stress-testing their efficacy, and maintaining meticulous risk management protocols.
This report outlines the safety work carried out for the OpenAI o1-preview and OpenAI o1-mini models, including safety evaluations, external red teaming, and Preparedness Framework evaluations.
"""

print("✔️ 데이터가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(system_prompt=system_prompt, user_input=data)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(system_prompt=system_prompt, user_input=data)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(system_prompt=system_prompt, user_input=data)
print(f"# Gemini Result: {gemini_response}")

▶︎ **실습문제**: **Translation_b**


<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 번역 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 원활한 실습을 위한 추가 셋업을 해봅시다. (pdf 관련 라이브러리 설치)

In [ ]:
%pip install PyPDF2

In [ ]:
import PyPDF2


def upload_pdf(file_path):
    """PDF 파일에서 텍스트 추출"""
    text = ""
    with open(file_path, "rb") as f:
        print(f"✅ 업로드 완료: {file_path}")
        for page in PyPDF2.PdfReader(f).pages:
            text += page.extract_text()
        print("\n================ PDF 내용 ==================")
        print(text[:1000])
        print("============================================\n")
    return text


def process_pdf_with_models(file_content, system_prompt):
    """로컬 PDF 파일로부터 텍스트 추출 및 모델 호출"""
    data = f"content = {file_content}"

    # OpenAI 요청
    openai_result = run_openai_chain(system_prompt=system_prompt, user_input=data)
    print(f"# OpenAI Result:\n{openai_result}")
    print("-" * 20)

✅ 이제 파일을 업로드하여 결과를 확인해봅시다! (only openai model)

In [ ]:
# 파일 업로드
# 실행: 테스트 파일 = Trump.pdf  -- 만약 오류가 난다면, 파일 경로 확인해주세요!!
pdf_path = "../03-프롬프트_분석/data/Trump.pdf"
pdf_content = upload_pdf(pdf_path)

In [ ]:
# 실행: 테스트 파일 = Trump.pdf  -- 만약 오류가 난다면, 파일 경로 확인해주세요!!
process_pdf_with_models(pdf_content, system_prompt)

▶︎ **실습문제**: **생소한 단어를 찾는 프롬프트 작성하기**

<hr>

✏️ **목표:** 단어 `whatpu` 와 `farduddle`의 의미를 찾는 프롬프트를 퓨삿을 사용하여 작성하기

📑 **출력 형태:** `단어: 뜻, 예시 문장`



<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=system_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=system_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=system_prompt)
print(f"# Gemini Result: {gemini_response}")

▶︎ **실습문제**: **다음 수학 문제를 Chain-of-thought 기법을 이용해 정답을 풀이해보세요**

<hr>

✏️ **조건:** CoT 방법론을 활용해서 아래 문제의 답을 구하기
```
이 그룹의 홀수들을 더하면 짝수가 됩니다: 4, 8, 9, 15, 12, 2, 1.
답: 거짓입니다.
이 그룹의 홀수들을 더하면 짝수가 됩니다: 17, 10, 19, 4, 8, 12, 24.
답: 참입니다.
이 그룹의 홀수들을 더하면 짝수가 됩니다: 16, 11, 14, 4, 8, 13, 24.
답: 참입니다.
이 그룹의 홀수들을 더하면 짝수가 됩니다: 17, 9, 10, 12, 13, 4, 2.
답: 거짓입니다.
이 그룹의 홀수들을 더하면 짝수가 됩니다: 15, 32, 5, 13, 82, 7, 1.
답:
```



<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=system_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=system_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=system_prompt)
print(f"# Gemini Result: {gemini_response}")

▶︎ **실습문제**: **Zero-shot CoT를 활용한 수학문제 풀이**

<hr>

✏️ **조건:** 수학 문제를 Zero-shot CoT를 사용해서 문제를 풀어보세요.
```
📌 문제

저는 시장에 가서 사과 10개를 샀습니다.
이웃에게 사과 2개를 주고 강아지에게도 2개를 줬습니다.
그리고 나서 5개의 사과를 더 사서 1개를 먹었습니다.
제가 가진 사과는 몇 개가 남았나요?
```



<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=system_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=system_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=system_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> Reasoning 기법: Chain-of-X 패러다임









▶︎ **실습문제**: **Chain-of-thought 기법을 사용하여 영어 학습 챗봇 만들기**

<hr>

> ✍️ **상황:** 영어 화자 David이 한국어를 앱을 통해 학습하는 상황입니다. Intermediate level의 학습자예요.
학생 이름은 David이고, 학습 주제는 “비즈니스 마케팅” 입니다.학습해야 하는 단어는 : 전략, 목표, 분석입니다.


> 🎯 **조건:** 1~3단계가 한 프롬프트에 담겨야 합니다.
- **1단계**: 각 단어의 뜻을 가르쳐주는 멀티턴 챗봇을 만드세요.
- **2단계**: 각 단어를 따라 읽도록하고, David 에게 예시 문장을 만들어보라고 하세요. 세 단어 반복해주세요.
- **3단계**: David 이 잘따라한다면 칭찬을 해주고, 잘 따라오지 못한다면 guide 를 주는 프롬프트를 제작하세요.

챗봇이 대화를 능동적으로 마무리 할 수 있도록 합니다.

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
provider = "openai"  # gpt-4o-mini
graph = create_chat_graph(provider, system_prompt)
print(f"💬 {provider.upper()} LangGraph 대화 시작 (종료: q or quit)\n")

thread_id = "1"

while True:
    user_input = input("prompt: ").strip()
    if user_input.lower() in ["q", "quit"]:
        print("대화를 종료합니다.")
        break

    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config={"configurable": {"thread_id": thread_id}},
    )

    print(f"{provider.upper()} → {result['messages'][-1].content}\n")

▶︎ **실습문제**: **번역을 위한 thought generation: step-back prompting, thread-of-thought 연습**

<hr>

> ✍️ **목표:** 사용 데이터는 2025년 미국 트럼프 대통령 취임사 전문입니다.
영한 번역을 human-level 의 완성도로 하는 프롬프트를 작성해보세요.


> 🎯 **조건:** `step-back prompting`과 `thread-of-thought` 기법을 사용하세요.

- [동아일보 기자가 작성한 번역문](https://www.donga.com/news/Inter/article/all/20250121/130901105/1)을 정답으로 간주합니다.

- Step-back prompting 기법으로 제작한 프롬프트의 결과를 정답과 비교해보세요.
- Thread-of-thought 기법으로 제작한 프롬프트의 결과를 정답과 비교해보세요.



<hr>
<details>
<summary>🔽 Step-back prompting 사용한 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>



<details>
<summary>🔽 Thread-of-prompting 사용한 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>


✏️ 문제의 요구사항에 따라 `step-back prompting` 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt_step_back = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ step-back prompting 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt_step_back)

✏️ 문제의 요구사항에 따라 `thread-of-thought` 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt_tot = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ thread-of-thought 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt_tot)

✅ 이제 원활한 실습을 위한 추가 셋업을 해봅시다. (pdf 관련 라이브러리 설치)

In [ ]:
%pip install PyPDF2

In [ ]:
import PyPDF2


def upload_pdf(file_path):
    """PDF 파일에서 텍스트 추출"""
    text = ""
    with open(file_path, "rb") as f:
        print(f"✅ 업로드 완료: {file_path}")
        for page in PyPDF2.PdfReader(f).pages:
            text += page.extract_text()
        print("\n================ PDF 내용 ==================")
        print(text[:1000])
        print("============================================\n")
    return text


def process_pdf_with_models(file_content, system_prompt):
    """로컬 PDF 파일로부터 텍스트 추출 및 모델 호출"""
    data = f"content = {file_content}"

    # OpenAI 요청
    openai_result = run_openai_chain(system_prompt=system_prompt, user_input=data)
    print(f"# OpenAI Result:\n{openai_result}")
    print("-" * 20)

✅ 이제 트럼프 취임사 전문 파일을 업로드하여 두 프롬프트 결과를 확인해봅시다! (only openai model)

In [ ]:
# 파일 업로드
# 실행: 테스트 파일 = Trump.pdf  -- 만약 오류가 난다면, 파일 경로 확인해주세요!!
pdf_path = "../03-프롬프트_분석/data/Trump.pdf"
pdf_content = upload_pdf(pdf_path)

In [ ]:
# Step back 프롬프트 결과 (실행 시간이 오래 걸릴 수도 있습니다.)
process_pdf_with_models(pdf_content, system_prompt_step_back)

In [ ]:
# Thread-of-thought 프롬프트 결과 (실행 시간이 오래 걸릴 수도 있습니다.)
process_pdf_with_models(pdf_content, system_prompt_tot)